# Modül 02: Aktivasyon Fonksiyonları
## Deep Learning Path — Kapsamlı Jupyter Notebook

> Tüm aktivasyon fonksiyonlarını matematiksel temelden görsel karşılaştırmaya kadar.

---
## 📋 İçindekiler
1. [Giriş ve Öğrenme Hedefleri](#1)
2. [Matematiksel Arka Plan — Formüller ve Türevler](#2)
3. [NumPy Implementasyonları](#3)
4. [Görsel Karşılaştırma](#4)
5. [Vanishing Gradient — Kanıt ve Görsel](#5)
6. [Dead Neuron Problemi](#6)
7. [TensorFlow/Keras](#7)
8. [PyTorch](#8)
9. [Hangi Aktivasyonu Ne Zaman Seç?](#9)
10. [Hiperparametre Deneyleri](#10)
11. [Özet ve Alıştırmalar](#11)


## 1. Öğrenme Hedefleri <a id='1'></a>
- ✅ 8 aktivasyon fonksiyonunun formülünü, türevini ve aralığını söylemek
- ✅ Vanishing gradient problemini matematiksel olarak kanıtlamak
- ✅ Dead Neuron problemini ve çözümlerini açıklamak
- ✅ Hangi durumda hangi aktivasyonu seçeceğini bilmek
- ✅ Tüm aktivasyonları NumPy, TensorFlow ve PyTorch ile kodlamak


## 2. Matematiksel Arka Plan <a id='2'></a>

| Fonksiyon | Formül | Türev | Aralık |
|-----------|--------|-------|--------|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma(z)(1-\sigma(z))$ | $(0,1)$ |
| Tanh | $\tanh(z)$ | $1-\tanh^2(z)$ | $(-1,1)$ |
| ReLU | $\max(0,z)$ | $\mathbb{1}[z>0]$ | $[0,+\infty)$ |
| Leaky ReLU | $\max(\alpha z, z)$ | $1$ veya $\alpha$ | $(-\infty,+\infty)$ |
| ELU | $z$ veya $\alpha(e^z-1)$ | $1$ veya $\text{ELU}+\alpha$ | $(-\alpha,+\infty)$ |
| SELU | $\lambda \cdot \text{ELU}(z,\alpha)$ | $\lambda \cdot \text{ELU türevi}$ | $(≈{-1.76},+\infty)$ |
| Swish | $z \cdot \sigma(z)$ | $\sigma(z) + z\sigma(z)(1-\sigma(z))$ | $(≈{-0.28},+\infty)$ |
| GELU | $z \cdot \Phi(z)$ | $\Phi(z) + z\phi(z)$ | $(≈{-0.17},+\infty)$ |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# ── 8 Aktivasyon Fonksiyonu ────────────────────────────────────
def sigmoid(z):    return 1/(1+np.exp(-np.clip(z,-500,500)))
def sigmoid_d(z):  s=sigmoid(z); return s*(1-s)
def tanh_fn(z):    return np.tanh(z)
def tanh_d(z):     return 1-np.tanh(z)**2
def relu(z):       return np.maximum(0,z)
def relu_d(z):     return (z>0).astype(float)
def lrelu(z,a=.01): return np.where(z>0,z,a*z)
def lrelu_d(z,a=.01): return np.where(z>0,1.,a)
def elu(z,a=1.):   return np.where(z>0,z,a*(np.exp(np.clip(z,-500,0))-1))
def elu_d(z,a=1.): return np.where(z>0,1.,elu(z,a)+a)
def selu(z):
    al,lm=1.6732632423543772,1.0507009873554804
    return lm*np.where(z>0,z,al*(np.exp(np.clip(z,-500,0))-1))
def selu_d(z):
    al,lm=1.6732632423543772,1.0507009873554804
    return lm*np.where(z>0,1.,al*np.exp(np.clip(z,-500,0)))
def swish(z):      return z*sigmoid(z)
def swish_d(z):    s=sigmoid(z); return s+z*s*(1-s)
def gelu(z):       return .5*z*(1+np.tanh(np.sqrt(2/np.pi)*(z+.044715*z**3)))
def gelu_d(z):     cdf=.5*(1+np.tanh(np.sqrt(2/np.pi)*(z+.044715*z**3))); return cdf+z*np.exp(-.5*z**2)/np.sqrt(2*np.pi)

ACTS = {
    'Sigmoid':    (sigmoid,sigmoid_d,'#1565C0'),
    'Tanh':       (tanh_fn,tanh_d,'#00695C'),
    'ReLU':       (relu,relu_d,'#E65100'),
    'Leaky ReLU': (lrelu,lrelu_d,'#6A1B9A'),
    'ELU':        (elu,elu_d,'#AD1457'),
    'SELU':       (selu,selu_d,'#2E7D32'),
    'Swish':      (swish,swish_d,'#F57F17'),
    'GELU':       (gelu,gelu_d,'#880E4F'),
}

z = np.linspace(-4, 4, 400)
z_test = np.array([-2.,-1.,0.,1.,2.])

print("✓ Tüm aktivasyon fonksiyonları tanımlandı")
print(f"  Test değerleri (z={list(z_test)}):")
print(f"  {'Fonksiyon':<14} " + "  ".join(f"z={v:>4.0f}" for v in z_test))
for name,(fn,_,_) in ACTS.items():
    print(f"  {name:<14} " + "  ".join(f"{v:>7.4f}" for v in fn(z_test)))


## 3. Görsel Karşılaştırma <a id='4'></a>

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16,8))
fig.suptitle('Aktivasyon Fonksiyonları — f(z) ve f\'(z)', fontsize=14, fontweight='bold')

for i, (name, (fn, dfn, color)) in enumerate(ACTS.items()):
    ax = axes[i//4][i%4]
    ax.plot(z, fn(z), color=color, lw=2.5, label='f(z)')
    ax.plot(z, dfn(z), color=color, lw=1.5, linestyle='--', alpha=0.6, label="f'(z)")
    ax.axhline(0, color='gray', lw=0.7, alpha=0.5)
    ax.axvline(0, color='gray', lw=0.7, alpha=0.5)
    ax.set_title(name, fontweight='bold', color=color)
    ax.set_xlim(-4,4); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    
plt.tight_layout()
plt.show()


## 4. Vanishing Gradient — Matematiksel Kanıt <a id='5'></a>

**Problem:** Sigmoid türevinin maksimumu 0.25'tir. $n$ katmanlı bir ağda:

$$\text{Gradyan} \approx (0.25)^n$$

| Katman | Sigmoid | Tanh | ReLU |
|--------|---------|------|------|
| 1 | 0.25 | 0.70 | 1.0 |
| 5 | 0.00098 | 0.168 | 1.0 |
| 10 | $9.5 \times 10^{-7}$ | 0.028 | 1.0 |
| 20 | $\approx 0$ | 0.0008 | 1.0 |

**ReLU'nun avantajı:** $z > 0$ için türev her zaman 1 → gradyan katman katman ölmüyor!


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: Katman sayısı ile gradyan yok oluşu
ax1 = axes[0]
layers = np.arange(1, 21)
ax1.semilogy(layers, 0.25**layers, 'b-o', label='Sigmoid (max 0.25)', lw=2, ms=5)
ax1.semilogy(layers, 0.7**layers,  'g-s', label='Tanh (tipik 0.7)',   lw=2, ms=5)
ax1.semilogy(layers, np.ones(20),  'r-^', label='ReLU (z>0 → 1.0)',  lw=2, ms=5)
ax1.axhline(1e-4, color='gray', linestyle=':', alpha=0.7, label='Eşik 1e-4')
ax1.fill_between(layers, 0, 1e-4, alpha=0.1, color='red')
ax1.set_xlabel('Katman Sayısı'); ax1.set_ylabel('Gradyan (log)')
ax1.set_title('Vanishing Gradient: Derinlikle Yok Oluş', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Sağ: Sigmoid türevi
ax2 = axes[1]
z_w = np.linspace(-8, 8, 400)
ax2.plot(z_w, sigmoid_d(z_w), 'b-', lw=2.5, label="σ'(z)")
ax2.fill_between(z_w, 0, sigmoid_d(z_w), alpha=0.2, color='blue')
ax2.axhline(0.25, color='red', linestyle='--', lw=1.5, label='Maks = 0.25')
ax2.axhspan(0, 0.01, alpha=0.15, color='red')
ax2.text(5, 0.005, 'Etkisiz\nbölge', color='red', fontsize=9)
ax2.set_title("Sigmoid Türevi — Çoğu z'de Sıfıra Yakın!", fontweight='bold')
ax2.set_xlabel('z'); ax2.set_ylabel("σ'(z)")
ax2.legend(); ax2.set_ylim(-0.02, 0.30); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("10 katmanlı ağda sigmoid gradyanı:", f"{0.25**10:.2e}")
print("10 katmanlı ağda ReLU gradyanı:   ", 1.0)


## 5. Dead Neuron Problemi <a id='6'></a>

ReLU'da z≤0 olan nöron türev=0 alır → ağırlıkları hiç güncellenmez → **ölü nöron**.

**Çözümler:** Leaky ReLU, ELU, öğrenme hızını düşürmek, He başlatma.

In [ ]:
# Dead neuron görselleştirmesi
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
z_d = np.linspace(-3, 3, 300)

for ax, (name, fn, color) in zip(axes, [
    ('ReLU', relu, '#E65100'),
    ('Leaky ReLU', lrelu, '#6A1B9A'),
    ('ELU', elu, '#AD1457'),
]):
    vals = fn(z_d)
    ax.plot(z_d, vals, color=color, lw=2.5)
    # Ölü bölgeyi işaretle (sadece ReLU için)
    if name == 'ReLU':
        ax.fill_between(z_d[z_d<0], 0, vals[z_d<0], alpha=0.3, color='red', label='Ölü bölge (grad=0)')
        ax.text(-2, -0.3, '⚠ ÖLÜM BÖLGE
grad = 0', color='red', fontsize=9, ha='center')
    ax.axhline(0, color='gray', lw=0.7)
    ax.axvline(0, color='gray', lw=0.7)
    ax.set_title(f'{name}', fontweight='bold', color=color)
    ax.set_xlim(-3,3); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle('Dead Neuron Problemi ve Çözümleri', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. TF/Keras ve PyTorch <a id='7'></a>

In [ ]:
# TensorFlow/Keras aktivasyon kullanımı
try:
    import tensorflow as tf
    z_tf = tf.constant([-2., -1., 0., 1., 2.])
    print("TensorFlow aktivasyon sonuçları:")
    print(f"  ReLU : {tf.keras.activations.relu(z_tf).numpy()}")
    print(f"  GELU : {tf.keras.activations.gelu(z_tf).numpy().round(4)}")
    print(f"  Swish: {tf.keras.activations.swish(z_tf).numpy().round(4)}")
    print()
    print("Keras'ta aktivasyon kullanımı:")
    print("  Dense(64, activation='relu')    # string")
    print("  Dense(64, activation='gelu')    # transformer için")
    print("  LeakyReLU(alpha=0.2)            # özel alpha")
except: print("TensorFlow yüklü değil")

try:
    import torch; import torch.nn as nn
    z_pt = torch.tensor([-2., -1., 0., 1., 2.])
    print("\nPyTorch aktivasyon sonuçları:")
    print(f"  ReLU : {nn.ReLU()(z_pt).numpy()}")
    print(f"  GELU : {nn.GELU()(z_pt).numpy().round(4)}")
    print(f"  SiLU : {nn.SiLU()(z_pt).numpy().round(4)}")  # Swish
    print(f"  ELU  : {nn.ELU()(z_pt).numpy().round(4)}")
except: print("PyTorch yüklü değil")


## 7. Karar Rehberi <a id='9'></a>

| Durum | Önerilen |
|-------|----------|
| Çıkış — ikili sınıflandırma | **Sigmoid** |
| Çıkış — çok sınıflandırma | **Softmax** |
| Çıkış — regresyon | Linear (aktivasyon yok) |
| Gizli — genel başlangıç | **ReLU** |
| Gizli — Transformer | **GELU** |
| Gizli — EfficientNet | **Swish** |
| Çok derin FC ağlar | **SELU** |
| Dead neuron sorunu var | **Leaky ReLU** veya **ELU** |
| RNN/LSTM | **Tanh** |


## 8. Alıştırmalar <a id='11'></a>

**1.** `parametric_relu(z, alpha)` fonksiyonu yaz: alpha öğrenilebilir parametre olsun.  
**2.** Sigmoid türevinin maksimumunun neden tam olarak 0.25 olduğunu matematik ile kanıtla.  
**3.** 20 katmanlı bir ağda tanh kullanılırsa kaçıncı katmandan itibaren gradyan 1e-6'nın altına düşer?  
**4.** ELU ve ReLU'yu aynı veri üzerinde eğit — hangi daha hızlı yakınsıyor?  
**5.** GELU'nun neden BERT'de tercih edildiğini araştır — stochastic regularization bağlantısını açıkla.

---
**Sonraki Modül:** Modül 03 — Kayıp Fonksiyonları ve Optimizasyon
